# Fig3c: combined OOD context figure

This notebook draws RPE1/K562 on the first row and Jurkat/HepG2 on the second row using the OOD `cluster_test` set.
Adjust the configuration section, then run all cells.


In [1]:
"""Plot the Fig3c OOD context-combined figure on one shared Matplotlib grid."""

from __future__ import annotations

import glob
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd


mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans", "Liberation Sans"]
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams.update(
    {
        "pdf.fonttype": 42,
        "font.size": 7,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.linewidth": 0.7,
    }
)


try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter does not define __file__; resolve from the repository root cwd.
    SCRIPT_DIR = Path.cwd()
    if SCRIPT_DIR.name != "fig3c":
        SCRIPT_DIR = SCRIPT_DIR / "figure" / "fig3c"
REPO_ROOT = SCRIPT_DIR.parents[1]
RESULTS_DIR = REPO_ROOT / "results" / "out_of_distribution"
OUT_STEM = SCRIPT_DIR / "fig3c_contexts_systema_correlation_systema_bias_all_combined"

GENE = "all"
SET_NAME = "cluster_test"
BUDGET_LEVELS = [0, 0.5, 1, 10, 30, 50, 80]
ERRORBAR = "std"

# These are the only layout controls needed for the combined figure.
FIGSIZE = (15, 7.5)
# Spacing between the four context panels.
CONTEXT_ROW_SPACING = 0.2
CONTEXT_COLUMN_SPACING = 0.1
# Spacing inside each context panel.
INTERNAL_ROW_SPACING = 0.08
INTERNAL_COLUMN_SPACING = 0.08
BOTTOM_MARGIN = 0.08
AXIS_LABEL_SIZE = 14
TICK_LABEL_SIZE = 12
X_TICK_ROTATION = 90
SHOW_X_LABEL = False

METRICS = ["systema_correlation", "systema_bias"]
CONTEXT_ORDER = [
    ("replogle_rpe1_essential_rpe1", "RPE1"),
    ("replogle_k562_essential_k562", "K562"),
    ("nadig_jurkat_essential_jurkat", "Jurkat"),
    ("nadig_hepg2_essential_hepg2", "HepG2"),
]
GROUPS = [
    ["CMean"],
    ["LM", "GEARS", "scGPT"],
    ["GenePert", "scLambda", "PRESAGE"],
    ["PMean"],
    ["scGen", "Biolord"],
]
ZEROSHOT_METHODS = {"PMean", "scGen", "Biolord"}
METHOD_COLORS = {
    "CMean": "#649553",
    "LM": "#FF8F2E",
    "GEARS": "#FC6500",
    "scGPT": "#D64101",
    "GenePert": "#F05A6F",
    "scLambda": "#C22A7B",
    "PRESAGE": "#7D1D67",
    "PMean": "#5A87B7",
    "scGen": "#B17CD5",
    "Biolord": "#54049E",
}


def save_pub_py(fig: mpl.figure.Figure, stem: Path, dpi: int = 600) -> None:
    """Save a figure as editable SVG and raster PNG.

    Args:
        fig: Figure to save.
        stem: Output path without a file extension.
        dpi: PNG resolution.
    """
    stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(f"{stem}.svg")
    fig.savefig(f"{stem}.png", dpi=dpi)
    print(f"saved -> {stem}.svg / {stem}.png")


def load_context(context: str) -> pd.DataFrame:
    """Load ALL-gene test metrics for one context."""
    methods = [method for group in GROUPS for method in group]
    frames = []
    for method in methods:
        pattern = RESULTS_DIR / context / method / "*" / "achievement_summary_rebuilt.csv"
        for path in glob.glob(str(pattern)):
            try:
                frame = pd.read_csv(path)
            except (OSError, pd.errors.EmptyDataError):
                continue
            selected = frame[
                frame["metric_name"].isin(METRICS)
                & (frame["gene"] == GENE)
                & (frame["set"] == SET_NAME)
            ]
            if not selected.empty:
                frames.append(selected)
    if not frames:
        raise ValueError(f"No metric rows found for context {context}.")
    return pd.concat(frames, ignore_index=True)


def snap_budgets(values: pd.Series) -> dict[float, float | None]:
    """Map raw budgets to nominal budget levels."""
    mapping = {}
    for value in values.astype(float).unique():
        nearest = min(BUDGET_LEVELS, key=lambda level: abs(value - level))
        tolerance = max(0.3, 0.15 * nearest)
        mapping[value] = nearest if abs(value - nearest) <= tolerance else None
    return mapping


def aggregate(df: pd.DataFrame, metric: str) -> dict[str, pd.DataFrame]:
    """Aggregate one metric over repeats for every method and budget."""
    selected = df[df["metric_name"] == metric].copy()
    selected["x"] = selected["x"].astype(float).map(snap_budgets(selected["x"]))
    selected = selected[selected["x"].notna()]
    curves = {}
    for method, method_df in selected.groupby("method"):
        curve = (
            method_df.groupby("x")["value"]
            .agg(mean="mean", std="std")
            .reset_index()
            .sort_values("x")
        )
        curve["err"] = curve["std"].fillna(0.0) if ERRORBAR != "none" else 0.0
        curves[method] = curve
    return curves


def plot_combined(context_tables: dict[str, pd.DataFrame]) -> mpl.figure.Figure:
    """Draw all contexts directly into one aligned 4-by-10 grid."""
    curves = {
        context: {metric: aggregate(table, metric) for metric in METRICS}
        for context, table in context_tables.items()
    }
    levels = sorted(
        {
            level
            for context in curves.values()
            for metric in context.values()
            for curve in metric.values()
            for level in curve["x"]
        }
    )
    positions = {level: index for index, level in enumerate(levels)}
    figure = plt.figure(figsize=FIGSIZE)
    outer_grid = figure.add_gridspec(
        2,
        2,
        hspace=CONTEXT_ROW_SPACING,
        wspace=CONTEXT_COLUMN_SPACING,
    )
    axes = [[None for _ in range(10)] for _ in range(4)]
    for context_index in range(4):
        outer_row, outer_column = divmod(context_index, 2)
        inner_grid = outer_grid[outer_row, outer_column].subgridspec(
            2,
            5,
            hspace=INTERNAL_ROW_SPACING,
            wspace=INTERNAL_COLUMN_SPACING,
        )
        for metric_index in range(2):
            for group_index in range(5):
                axes[outer_row * 2 + metric_index][outer_column * 5 + group_index] = figure.add_subplot(
                    inner_grid[metric_index, group_index]
                )
    shared_y = {metric: None for metric in METRICS}

    for context_index, (context, _) in enumerate(CONTEXT_ORDER):
        row_offset = 0 if context_index < 2 else 2
        show_x_ticks = context_index >= 2
        for metric_index, metric in enumerate(METRICS):
            row = row_offset + metric_index
            for group_index, group in enumerate(GROUPS):
                column = context_index % 2 * 5 + group_index
                axis = axes[row][column]
                if shared_y[metric] is None:
                    shared_y[metric] = axis
                else:
                    axis.sharey(shared_y[metric])
                axis.axhline(0, color="#B0B0B0", linewidth=0.6, zorder=0)
                for method in [name for name in group if name in curves[context][metric]]:
                    curve = curves[context][metric][method]
                    if method not in ZEROSHOT_METHODS:
                        curve = curve[curve["x"] != 0]
                    xs = curve["x"].map(positions).to_numpy()
                    color = METHOD_COLORS[method]
                    axis.plot(
                        xs,
                        curve["mean"],
                        marker="o",
                        markersize=3.2,
                        linewidth=1.5,
                        color=color,
                        clip_on=False,
                        zorder=3,
                    )
                    if ERRORBAR != "none":
                        axis.fill_between(
                            xs,
                            curve["mean"] - curve["err"],
                            curve["mean"] + curve["err"],
                            color=color,
                            alpha=0.16,
                            linewidth=0,
                            zorder=2,
                        )
                axis.set_xticks(range(len(levels)))
                axis.set_xticklabels(
                    [f"{level:g}" for level in levels] if show_x_ticks and metric_index == 1 else [],
                    fontsize=TICK_LABEL_SIZE,
                    rotation=X_TICK_ROTATION,
                    ha="center",
                )
                axis.yaxis.set_major_locator(
                    MaxNLocator(nbins=2, steps=[1, 2, 3, 5, 10], min_n_ticks=3)
                )
                axis.tick_params(
                    length=2.5,
                    width=0.6,
                    labelsize=TICK_LABEL_SIZE,
                    labelbottom=show_x_ticks and metric_index == 1,
                    labelleft=group_index == 0,
                )
                axis.grid(True, axis="y", alpha=0.2, linewidth=0.5, zorder=0)
                axis.set_axisbelow(True)

    if SHOW_X_LABEL:
        figure.supxlabel("Query-context data availability (%)", fontsize=AXIS_LABEL_SIZE)
    figure.subplots_adjust(left=0.04, right=0.995, bottom=BOTTOM_MARGIN, top=0.995)
    return figure


def main() -> None:
    """Load all contexts and export only the combined SI5 figure."""
    if not RESULTS_DIR.is_dir():
        raise FileNotFoundError(f"Results directory does not exist: {RESULTS_DIR}")
    tables = {context: load_context(context) for context, _ in CONTEXT_ORDER}
    figure = plot_combined(tables)
    save_pub_py(figure, OUT_STEM)
    plt.close(figure)


if __name__ == "__main__":
    main()
